![Gradient Boosting Validation — governed local experiment using grouped inputs, a fixed model, validation-only evidence, comparison, and explicit limitations](../docs/assets/ecg-gradient-boosting-validation-banner.png)

# Step 2 — High-performing gradient boosting validation example — v3.88

Welcome! This notebook loads the governed train and validation artifacts from Step 0, fits one fixed gradient-boosting model in memory, and walks through validation-only metrics and limitations. It does not tune a model, open the protected test partition, or save a model artifact.

Once [`Step 0`](./00-environment-setup-and-artifact-generation.ipynb) artifacts exist, plan on roughly **5–15 minutes** on a typical laptop. Reruns usually take a similar amount of time because the model is refit in memory; generated data size, CPU, memory, and system load vary. Minute-scale heartbeats keep the fit visibly active without pretending to predict its finish time.

**Jump to:** [prerequisite check](#01-stop--complete-step-0-first) · [load waveform windows](#4-load-raw-waveform-windows) · [train model](#5-train-fixed-tuned-gradient-boosting-model) · [validation metrics](#6-validation-only-metrics) · [limitations](#9-lineage-and-limitations) · [version appendix](#appendix-version-history-and-change-log)

The [environment reproducibility guide](../docs/environment-reproducibility.md) documents the supported notebook environment. Recommended order: [`Step 0` setup](./00-environment-setup-and-artifact-generation.ipynb) → [`Step 1` narrative walkthrough](./01-narrative-walkthrough.ipynb) → Step 2 model validation example.

<div role="note" aria-label="Research and educational use boundary" style="border: 3px solid #007c91; border-left-width: 8px; border-radius: 6px; background: #eefbff; color: #102a43; padding: 1rem 1.25rem; margin: 1rem 0; line-height: 1.5;"><strong style="display: block; font-size: 1.15rem; margin-bottom: 0.35rem;">Research and educational use boundary</strong>This is local validation-only experimentation. It is not clinical ECG software, diagnostic software, patient-monitoring software, production ML, benchmark evidence, or proof of deployment fitness.</div>



## 0. Confirm the supported local checkout

Run this cell first. It confirms that the notebook is using the repository checkout and locked project kernel, then keeps notebook 00 generated state in place for the prerequisite checks below. No data or artifacts are copied. Optional web-hosted execution is deferred to [Issue #200](https://github.com/Jared-Godar/ecg_anomaly_detection/issues/200).


> **CODE CELL FUNCTION**
>
> Confirm the supported local checkout and project kernel before reading Step 0 state.


In [ ]:
# CODE CELL FUNCTION:
# Confirm the supported local checkout and project kernel before downstream work.
#
# COMMENT MAP:
# - Locate the repository root even when Jupyter starts inside notebooks/.
# - Require the locked project kernel selected during local setup.
# - Reuse ignored Step 0 artifacts in place without copying generated state.
# - Keep the protected test partition unopened.

from __future__ import annotations

import os
import sys
from pathlib import Path

# Jupyter may start with notebooks/ as its working directory, so walk upward to
# find the two stable repository markers used throughout the curated workflow.
repository_root = None
# Inspect candidates from nearest to farthest so this checkout wins deterministically.
for candidate in (Path.cwd(), *Path.cwd().parents):
    # Requiring both markers avoids adopting an unrelated Python-project ancestor.
    if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
        repository_root = candidate.resolve()
        break
# Continuing outside the checkout would make every relative artifact path ambiguous.
if repository_root is None:
    raise RuntimeError("Repository root not found. Open this notebook from the local project checkout.")

# The supported workflow uses the checkout-local environment created by uv. Compare
# the interpreter path lexically: uv creates the venv with the interpreter symlinked
# to a base Python outside .venv, so resolving it would falsely reject the correctly
# selected kernel. This gate still gives a direct remediation before a later import error.
expected_kernel_directory = (repository_root / ".venv").absolute()
# Keep the displayed executable path without resolving through the venv symlink.
kernel_executable = Path(sys.executable).absolute()
# Stop before imports when the editor selected a global or unrelated kernel.
if not kernel_executable.is_relative_to(expected_kernel_directory):
    raise RuntimeError(
        "Select the checkout-local .venv Python kernel before continuing.\n"
        f"Expected beneath: {expected_kernel_directory}\n"
        f"Current kernel: {kernel_executable}"
    )

# Anchor all later repository-relative paths to the verified local checkout.
os.chdir(repository_root)
# Publish a simple gate that later review/debug cells can inspect in memory.
CONTINUITY_READY = True
# Keep the local no-copy and protected-partition contract visible to the user.
NOTEBOOK_CONTINUITY = {
    "status": "local_checkout_ready",
    "execution_environment": "local checkout",
    "action": "reuse_generated_state_in_place",
    "repository_root": str(repository_root),
    "generated_state_copied": False,
    "protected_test_partition_opened": False,
}
print("[continuity] Local checkout and project kernel confirmed; Step 0 state remains in place.", flush=True)

NOTEBOOK_CONTINUITY


## 0.1 Stop — complete Step 0 first

<div role="alert" aria-label="Step 0 prerequisite" style="border: 3px solid #b42318; border-left-width: 8px; border-radius: 6px; background: #fff4f2; color: #5c160f; padding: 1rem 1.25rem; margin: 1rem 0; line-height: 1.5;"><strong style="display: block; font-size: 1.15rem; margin-bottom: 0.35rem;">Stop: complete Step 0 before training</strong>Run <a href="./00-environment-setup-and-artifact-generation.ipynb">00-environment-setup-and-artifact-generation.ipynb</a> first if you have not already generated the governed local artifacts.</div>

Step 0 handles environment setup, launch options, local artifact preflight, governed pipeline execution, and post-pipeline verification.

Open Step 0 here:

[`00-environment-setup-and-artifact-generation.ipynb`](00-environment-setup-and-artifact-generation.ipynb)

This notebook still includes a lightweight prerequisite/status suite below. If Step 0 has not fully generated model-ready artifacts, the suite reports a controlled blocked state instead of allowing a later cryptic file-loading error.

Then read the supported modernization walkthrough if you want the repository lifecycle context:

[`01-narrative-walkthrough.ipynb`](01-narrative-walkthrough.ipynb)


> **CODE CELL FUNCTION**
>
> Run a lightweight prerequisite/status suite before importing modeling dependencies or loading generated artifacts. This cell is strict: Step 2 stops when Step 0 has not produced model-ready artifacts or when the active kernel cannot import required modeling dependencies.


In [ ]:
# CODE CELL FUNCTION:
# Enforce Step 0 artifact readiness before the model validation example runs.
#
# COMMENT MAP:
# - Treat Step 2 as a real validation example, not a graceful-skip report.
# - Require the uv-backed notebook kernel to expose all modeling dependencies.
# - Require Step 0 status to be complete and verify the exact artifacts Step 2 loads.
# - Fail fast when PIPE-006 or any other Step 0 failure prevents model-ready artifacts.
# - Keep the protected test partition unopened; only train/validation readiness is checked.
# - Initialize shared paths only after the prerequisite contract is satisfied.

from __future__ import annotations

import importlib.util
import json
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Any

# Referenced only in failure/guidance messages below, not executed by this notebook.
STEP_0_NOTEBOOK = "notebooks/00-environment-setup-and-artifact-generation.ipynb"
# Same as STEP_0_NOTEBOOK above: a reference path for guidance messages only.
STEP_1_NOTEBOOK = "notebooks/01-narrative-walkthrough.ipynb"
# Flipped to True only after every readiness check below passes; every early return
# path in this cell is a raise, so a False value here never reaches later cells.
STEP2_READY = False
# Reserved for future soft-failure reporting; this cell currently raises on any
# blocking condition rather than accumulating blockers, so this stays empty.
STEP2_BLOCKERS: list[str] = []
# Reserved for future non-fatal notices; unused by this cell's current strict checks.
STEP2_WARNINGS: list[str] = []
# Machine-readable readiness summary for later cells/display; test_partition_opened
# is asserted False here and again at the bottom of this cell as an explicit,
# doubly-recorded guarantee that Step 2 never reads the protected test partition.
STEP2_CONTEXT: dict[str, object] = {
    "repository_root": None,
    "dataset_index": None,
    "run_id": None,
    "train_validation_artifacts_ready": False,
    "test_partition_opened": False,
}


def discover_repository_root(start: Path) -> Path:
    """Return the nearest repository root or raise before any model work begins."""
    # Walk upward from the kernel's working directory rather than assuming Path.cwd()
    # is already the repository root, since notebook kernels can be launched from a
    # subdirectory.
    for candidate in [start, *start.parents]:
        # Both markers together distinguish the real repository root from any
        # unrelated ancestor directory that happens to contain one but not the other.
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Repository root was not found. Open Step 2 from inside the cloned repository.")


def load_json_object(path: Path) -> dict[str, Any]:
    """Load one JSON object and reject missing, malformed, or non-object payloads."""
    # A missing file means the artifact this caller depends on was never generated.
    if not path.is_file():
        raise RuntimeError(f"Required artifact is missing: {path}")
    # A file that fails to parse means it was corrupted or hand-edited.
    try:
        value = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Invalid JSON artifact: {path}") from exc
    # Every artifact this function is used for is expected to be a JSON object, not
    # a bare list/scalar; catching that shape mismatch here keeps callers simple.
    if not isinstance(value, dict):
        raise RuntimeError(f"Expected JSON object artifact: {path}")
    return value


def relative_to_root(path: Path, root: Path) -> str:
    """Render a repository-relative path when possible for readable failure output."""
    # A path outside root (unlikely here, but possible for a misconfigured root)
    # falls back to its absolute form rather than raising from a display helper.
    try:
        return path.relative_to(root).as_posix()
    except ValueError:
        return path.as_posix()


def require_modules(module_names: list[str]) -> None:
    """Require notebook/modeling dependencies before any validation cells run."""
    missing = [name for name in module_names if importlib.util.find_spec(name) is None]
    # Fail with the specific missing package names rather than letting the first
    # subsequent import statement raise an opaque ModuleNotFoundError mid-cell.
    if missing:
        raise RuntimeError(
            "Step 2 cannot run the validation example because the active kernel is missing dependencies: "
            + ", ".join(missing)
            + ". Run Step 0 environment bootstrap and select the uv-backed notebook kernel."
        )


def require_step0_complete(root: Path) -> dict[str, Any]:
    """Require strict Step 0 success instead of accepting blocked status as readiness."""
    status_path = root / "notebooks" / "local" / "step0-pipeline-status.json"
    status = load_json_object(status_path)
    # A blocked/failed status must raise here rather than let Step 2 proceed against
    # artifacts that were never actually generated.
    if status.get("status") != "complete":
        raise RuntimeError(
            "Step 2 requires Step 0 to complete model-ready artifact generation. Step 0 did not complete.\n"
            f"Status: {status.get('status')}\n"
            f"Reason: {status.get('reason')}\n"
            f"Message: {status.get('message')}\n"
            f"Log file: {status.get('log_file')}"
        )
    return status


def newest_dataset_index(root: Path) -> Path:
    """Return the newest generated dataset index produced by Step 0 or raise."""
    candidates = sorted(
        (root / "data" / "processed" / "runs").glob("*/dataset-index.json"),
        key=lambda path: path.stat().st_mtime,
    )
    # No dataset index anywhere means Step 0 never reached the indexing stage,
    # regardless of what its own status file claims.
    if not candidates:
        raise RuntimeError(
            "Step 2 cannot run because Step 0 did not produce "
            "data/processed/runs/<run-id>/dataset-index.json."
        )
    return candidates[-1]


# Every later path in this cell (and later cells) is anchored off this one root.
root = discover_repository_root(Path.cwd())
# Record it in the shared readiness summary immediately, before any later check
# that might raise, so a partial failure still leaves this field populated.
STEP2_CONTEXT["repository_root"] = str(root)

# Modeling/plotting dependencies live in the notebooks/dev dependency group; a
# kernel launched outside `uv run --group notebooks jupyter lab` would otherwise
# fail with a confusing ImportError deep inside a later cell instead of here.
require_modules(["numpy", "sklearn", "matplotlib", "IPython", "ecg_anomaly_detection"])
# The supported notebook environment runs the package CLI through the checkout's
# locked project environment; this avoids dependence on whichever Python launched Jupyter.
if shutil.which("uv") is None:
    raise RuntimeError("`uv` is not available on PATH; complete Step 0 environment bootstrap first.")
# Exercise the same local project entry point used by Step 0 before loading any shards.
cli_command = ["uv", "run", "ecg-data", "--help"]
# Capture only failure diagnostics because successful help text is not user-facing evidence.
cli_check = subprocess.run(
    cli_command,
    cwd=root,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
    check=False,
)
# A nonzero exit here means the environment is broken in a way require_modules'
# import-only check couldn't detect (e.g. a package installed but misconfigured).
if cli_check.returncode != 0:
    raise RuntimeError("Project CLI verification failed; Step 2 is not running in the selected supported environment.\n" + cli_check.stderr[-1000:])

# A "complete" status alone is not sufficient proof; re-derive and re-verify the
# actual generated artifacts independently of what the status file claims.
step0_status = require_step0_complete(root)
# The freshest dataset index is what determines which run_id (and therefore which
# artifacts) the rest of this cell verifies.
index_path = newest_dataset_index(root)
# The dataset index's own parent directory name is the run ID that produced it.
run_id = index_path.parent.name
# The remaining required artifacts live in this run's artifacts/runs/<run-id>/ tree.
run_artifact_root = root / "artifacts" / "runs" / run_id
# The exact set of generated files this validation example depends on, labeled for
# the readable missing-artifact report below.
required_artifacts = {
    "dataset_index": index_path,
    "split_manifest": run_artifact_root / "split.json",
    "split_quality": run_artifact_root / "split_quality_summary.json",
}
# Verify every required file actually exists on disk, not just that the status file
# reported success.
missing = [f"{label}: {relative_to_root(path, root)}" for label, path in required_artifacts.items() if not path.is_file()]
# Any missing artifact makes the validation example unsafe to run.
if missing:
    raise RuntimeError("Step 2 cannot run because required Step 0 artifacts are missing:\n" + "\n".join(missing))

# The dataset index's own contents (partition/shard structure), not just its
# presence, must be verified before later cells trust it.
index_probe = load_json_object(index_path)
# Extracted once so both the shape check below and the per-partition loop after it
# read the same value.
partitions = index_probe.get("partitions")
# The dataset index always records all three partitions' shard locations, even
# though this cell (and this notebook) only ever loads train/validation below --
# the shape check here is about index integrity, not about opening every partition.
if not isinstance(partitions, dict) or set(partitions) != {"train", "validation", "test"}:
    raise RuntimeError("Dataset index must expose exactly train, validation, and test partitions.")
# Deliberately iterates only ("train", "validation"), never "test": this notebook
# must not open the protected held-out test partition (see docs/evaluation-policy.md
# and docs/benchmark-governance.md). Omitting "test" here is the enforcement point.
for partition_name in ("train", "validation"):
    partition = partitions.get(partition_name, {})
    shards = partition.get("shards", [])
    # A partition with zero shards would make Step 2's later training/validation
    # cells silently operate on no data instead of failing loudly here.
    if not isinstance(shards, list) or not shards:
        raise RuntimeError(f"Dataset index partition `{partition_name}` must expose at least one shard.")
    missing_shards: list[str] = []
    # Verify every shard file referenced by the index actually exists on disk,
    # rather than discovering a missing shard mid-load in a later cell.
    for shard in shards:
        relative = (shard.get("file") or {}).get("path")
        # Missing path metadata and a missing file on disk are both unusable; both
        # get recorded so the raised error lists every problem shard together.
        if not relative or not (root / relative).is_file():
            missing_shards.append(str(relative))
    # Report accumulated shard problems once per partition instead of failing on
    # the first missing shard, so the reviewer sees the full extent of the gap.
    if missing_shards:
        raise RuntimeError(f"Missing {partition_name} shard files:\n" + "\n".join(missing_shards[:20]))

# Reached only if every check above passed without raising.
STEP2_READY = True
STEP2_CONTEXT.update({
    "dataset_index": relative_to_root(index_path, root),
    "run_id": run_id,
    "train_validation_artifacts_ready": True,
    "test_partition_opened": False,
})

{
    "step_2_ready": STEP2_READY,
    "step_0_status": step0_status.get("status"),
    "run_id": run_id,
    "dataset_index": relative_to_root(index_path, root),
    "split_manifest": relative_to_root(run_artifact_root / "split.json", root),
    "split_quality": relative_to_root(run_artifact_root / "split_quality_summary.json", root),
    "test_partition_opened": False,
}


## 1. Model notebook flow

After the Step 0 prerequisite suite passes, this notebook does only the streamlined validation workflow:

1. discover repository-relative generated artifacts
2. validate grouped train/validation/test split metadata
3. load train and validation waveform windows
4. train one fixed `HistGradientBoostingClassifier`
5. report validation-only metrics
6. summarize lineage and limitations

<div role="alert" aria-label="Protected test boundary" style="border: 3px solid #b42318; border-left-width: 8px; border-radius: 6px; background: #fff4f2; color: #5c160f; padding: 1rem 1.25rem; margin: 1rem 0; line-height: 1.5;"><strong style="display: block; font-size: 1.15rem; margin-bottom: 0.35rem;">Protected test remains unopened</strong>The protected test partition is indexed for governance context but is not opened, scored, or reported. See the <a href="../docs/evaluation-policy.md">evaluation policy</a> and <a href="../docs/benchmark-governance.md">benchmark governance</a> boundary.</div>


> **CODE CELL FUNCTION**
>
> Import modeling dependencies only after the readiness/status cell has run. Missing imports are converted into additional blocked-state messages so the notebook can still finish with actionable Step 0 guidance.


In [ ]:
# COMMENT MAP:
# - Import display, progress, and modeling helpers only after the strict prerequisite gate passes.
# - Do not catch ImportError here; Step 2 is supposed to fail if the environment is wrong.
# - Keep imports explicit so reviewers can see the modeling stack used by the example.
# - Configure one shared stdout progress reporter for three potentially long phases.
# - Preserve validation-only scope: metrics are imported for validation, not protected test evaluation.

import hashlib
from collections import Counter
from typing import Any

from IPython.display import Markdown as _Markdown
from IPython.display import display as _display
from ecg_anomaly_detection.progress import ProgressReporter
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_sample_weight


def display_markdown(text: str) -> None:
    """Render Markdown in notebook environments without hiding dependency failures."""
    _display(_Markdown(text))


# Send immediately flushed, observational-only stage lines to the active notebook
# output stream. The reporter never changes model control flow or artifact contents.
NOTEBOOK_PROGRESS = ProgressReporter(stream=sys.stdout)

{
    "imports_ready": True,
    "step_2_ready": STEP2_READY,
    "modeling_stack": ["numpy", "matplotlib", "sklearn"],
}


> **CODE CELL FUNCTION**
>
> Resolve repository paths through the installed package when available. If Step 2 is blocked, this cell reports the blocked state and avoids package-backed assertions.


In [ ]:

# COMMENT MAP:
# - Prefer the package-backed RepositoryPaths helper when the environment is ready.
# - Do not assert or crash when Step 0 has not prepared the kernel; report blocked state instead.
# - Keep `root` stable from the prereq cell so later repository-relative diagnostics still work.
# - Avoid mutating files; this is environment orientation only.

# Assign one summary in every branch, then leave it as the final expression so
# VS Code and Jupyter visibly confirm successful path resolution.
if not STEP2_READY:
    # The prereq/import cells already captured the reason. Repeating it here keeps execution linear.
    repository_paths_summary = {
        "repository_paths_ready": False,
        "reason": "Step 2 is blocked before package-backed path resolution.",
        "blockers": STEP2_BLOCKERS,
    }
else:
    from ecg_anomaly_detection.config import RepositoryPaths

    paths = RepositoryPaths.discover()
    root = paths.root

    # Convert unexpected path mismatch into a blocker instead of an assertion wall.
    if not (root / "pyproject.toml").is_file():
        STEP2_READY = False
        STEP2_BLOCKERS.append("Repository root must contain pyproject.toml.")
        repository_paths_summary = {
            "repository_paths_ready": False,
            "blockers": STEP2_BLOCKERS,
        }
    else:
        repository_paths_summary = {
            "repository_paths_ready": True,
            "repository_root": str(root),
            "python": sys.executable,
            "package_backed": True,
        }

# A named final expression renders in notebook frontends; branch-local dict
# literals alone are swallowed because the compound ``if`` is the cell's last statement.
repository_paths_summary


> **CODE CELL FUNCTION**
>
> Define JSON-loading, local-state preflight, remediation-message, and artifact-discovery helpers with actionable missing-artifact messages.


In [ ]:

# COMMENT MAP:
# - Define helper functions for JSON loading, artifact readiness, and path remediation.
# - Keep helpers side-effect free except for reading local files.
# - Allow helpers to raise precise exceptions only when Step 2 is already ready and a contract is broken.
# - Keep remediation text focused on Step 0 because this notebook should not clean or rebuild artifacts.
# - Maintain protected-test discipline by requiring only train/validation artifacts downstream.

def load_json(path: Path) -> dict[str, Any]:
    """Load a JSON object from disk and reject non-object payloads.

    This helper is used after readiness checks pass. A non-object payload means the
    generated artifact contract is invalid and should be surfaced clearly.
    """
    # Stream from an open handle rather than read_text() + json.loads() so a large
    # artifact is parsed without holding two copies of its text in memory at once.
    with path.open(encoding="utf-8") as source:
        value = json.load(source)
    # Every artifact this helper loads is expected to be a JSON object, not a bare
    # list/scalar; catching that shape mismatch here keeps callers simple.
    if not isinstance(value, dict):
        raise ValueError(f"Expected a JSON object: {path}")
    return value


def local_clean_rebuild_message(*, repository_root: Path) -> str:
    """Return safe remediation guidance for rebuilding ignored local pipeline outputs.

    The command text is displayed only as guidance. This notebook never runs destructive
    cleanup automatically.
    """
    return (
        "Suggested remediation from the repository root:\n\n"
        "Run Step 0 first:\n"
        "  notebooks/00-environment-setup-and-artifact-generation.ipynb\n\n"
        "Step 0 owns safe local cleanup, governed acquisition, pipeline execution, "
        "and blocked-state diagnosis. Do not commit generated `data/` or `artifacts/` outputs."
    )


def missing_artifact_message(missing_path: Path, *, repository_root: Path) -> str:
    """Build a clear remediation message for missing generated artifacts."""
    relative = missing_path.relative_to(repository_root) if missing_path.is_relative_to(repository_root) else missing_path
    return (
        f"Required generated artifact is missing: {relative}\n\n"
        "Generated pipeline artifacts are intentionally ignored by Git, so a fresh clone will not include them.\n"
        "Complete Step 0 first: `notebooks/00-environment-setup-and-artifact-generation.ipynb`.\n\n"
        + local_clean_rebuild_message(repository_root=repository_root)
    )


def require_file(path: Path, *, repository_root: Path, purpose: str) -> Path:
    """Return a required file path or raise a precise artifact-contract error."""
    # A missing file here means the artifact contract was broken after the cell-0
    # readiness check passed (e.g. a file deleted between cells); report it with the
    # same remediation guidance as the original readiness check.
    if not path.is_file():
        raise FileNotFoundError(purpose + "\n\n" + missing_artifact_message(path, repository_root=repository_root))
    return path


def latest_dataset_index(repository_root: Path) -> Path:
    """Return the newest generated dataset index for local validation experimentation."""
    processed_runs_dir = repository_root / "data" / "processed" / "runs"
    candidates = sorted(
        processed_runs_dir.glob("*/dataset-index.json"),
        key=lambda path: path.stat().st_mtime,
    )
    # No dataset index anywhere means Step 0 never reached the indexing stage.
    if not candidates:
        raise FileNotFoundError(
            "No generated dataset index was found.\n\n"
            + missing_artifact_message(processed_runs_dir / "<run-id>" / "dataset-index.json", repository_root=repository_root)
        )
    return candidates[-1]


{
    "helper_functions_loaded": True,
    "side_effects": "none; helpers only read local artifacts when called",
    "step_2_ready": STEP2_READY,
}


> **CODE CELL FUNCTION**
>
> Resolve the current generated run, required manifests, and optional baseline metrics.


In [ ]:

# COMMENT MAP:
# - Select the latest generated dataset index only when prerequisites are ready.
# - Keep all run-specific artifact paths tied to the same run identifier.
# - Treat baseline metrics as optional because Step 2 can report validation metrics without them.
# - Avoid undefined variables in blocked mode by initializing placeholders.

index_path = None
# Populated once index_path is resolved, in the ready branch below.
run_id = None
# Populated once run_id is resolved, in the ready branch below.
run_artifact_root = None
# Populated alongside run_artifact_root, in the ready branch below.
split_path = None
# Populated alongside run_artifact_root, in the ready branch below.
split_quality_path = None
# Populated alongside run_artifact_root, in the ready branch below; unlike the
# other artifacts this one is optional, so its presence is only ever checked, not
# required (see baseline_metrics_available below).
baseline_metrics_path = None
# Populated by loading index_path's contents, in the ready branch below.
index = {}
# Populated by loading split_path's contents, in the ready branch below.
split_manifest = {}
# Populated by loading split_quality_path's contents, in the ready branch below.
split_quality = {}
# Reported in this cell's final summary regardless of which branch runs below.
baseline_metrics_available = False

# STEP2_READY is only True if cell 0's readiness gate passed without raising; this
# cell still guards against the (unreachable in normal use, but defensive) case of
# being re-run after that gate failed.
if not STEP2_READY:
    # Step 0 may have ended in a controlled blocked state such as PIPE-006.
    # In that case, Step 2 should complete as a readable blocked-state report.
    raise RuntimeError("Step 2 strict prerequisite gate failed before artifact loading: " + "; ".join(STEP2_BLOCKERS))
else:
    # The newest dataset index is treated as the local run selected for this notebook session.
    index_path = latest_dataset_index(root)
    run_id = index_path.parent.name

    # Related run artifacts are expected to share the same run identifier.
    run_artifact_root = root / "artifacts" / "runs" / run_id
    split_path = run_artifact_root / "split.json"
    split_quality_path = run_artifact_root / "split_quality_summary.json"
    baseline_metrics_path = run_artifact_root / "evaluation" / "validation-metrics.json"

    # Required artifacts are loaded or rejected with actionable messages.
    index = load_json(require_file(index_path, repository_root=root, purpose="Dataset index is required before loading train/validation windows."))
    split_manifest = load_json(require_file(split_path, repository_root=root, purpose="Split manifest is required before validating grouped partition lineage."))
    split_quality = load_json(require_file(split_quality_path, repository_root=root, purpose="Split quality summary is required before describing split validation status."))

    # Baseline metrics are useful for comparison but not required for this notebook to run.
    baseline_metrics_available = baseline_metrics_path.is_file()

{
    "step_2_ready": STEP2_READY,
    "run_id": run_id,
    "dataset_index": str(index_path.relative_to(root)) if index_path else None,
    "split_manifest": str(split_path.relative_to(root)) if split_path else None,
    "split_quality": str(split_quality_path.relative_to(root)) if split_quality_path else None,
    "baseline_metrics_available": baseline_metrics_available,
}


## 3. Validate grouped split assumptions

This check verifies the high-level grouping contract from the dataset index before any waveform arrays are opened.

It confirms:

- expected partition names exist
- subjects do not cross partitions
- records do not cross partitions
- the protected test partition remains indexed for lineage but is not opened for scoring


> **CODE CELL FUNCTION**
>
> Validate dataset-index partition structure and confirm train/validation/test grouping boundaries.


In [ ]:

# COMMENT MAP:
# - Validate grouped split assumptions only after generated manifests are loaded.
# - Confirm train, validation, and protected test partitions exist in metadata.
# - Check subject and record disjointness without opening protected test waveform arrays.
# - Keep blocked mode non-throwing so public execution can finish with a clear status.

expected_partitions = {"train", "validation", "test"}
# Populated from index.get("partitions") in the ready branch below.
partitions = {}
# Populated by partition_subjects() for every expected partition, in the ready branch below.
subject_sets = {}
# Populated by partition_records() for every expected partition, in the ready branch below.
record_sets = {}
# Reported in this cell's final summary regardless of which branch runs below.
split_status = "not evaluated"

# Mirrors cell 4's own STEP2_READY guard: this cell must not run split validation
# against artifacts that were never confirmed to exist.
if not STEP2_READY:
    raise RuntimeError("Step 2 is not ready; strict prerequisite gate did not pass, so split validation must not run.")
else:
    partitions = index.get("partitions")

    # The governed split contract should expose exactly these three partitions.
    if not isinstance(partitions, dict) or set(partitions) != expected_partitions:
        raise ValueError("Dataset index must contain exactly train, validation, and test partitions")


    def partition_subjects(name: str) -> set[str]:
        """Extract subject identifiers for one partition."""
        value = partitions[name]
        subjects = value.get("subject_ids", [])
        # A malformed subject_ids field would silently produce an empty/wrong set
        # below rather than surfacing the actual index corruption.
        if not isinstance(subjects, list):
            raise ValueError(f"Partition {name} has invalid subject_ids")
        return set(str(subject) for subject in subjects)


    def partition_records(name: str) -> set[str]:
        """Extract record identifiers from one partition's shard descriptors."""
        shards = partitions[name].get("shards", [])
        # Same defensive shape check as partition_subjects above, for shards.
        if not isinstance(shards, list):
            raise ValueError(f"Partition {name} has invalid shards")
        records = {str(shard.get("record_id")) for shard in shards}
        # An empty or "None" record ID means a shard descriptor is missing its
        # record_id field; that's an index-integrity bug, not a legitimate record.
        if "" in records or "None" in records:
            raise ValueError(f"Partition {name} has invalid record IDs")
        return records


    subject_sets = {name: partition_subjects(name) for name in expected_partitions}
    record_sets = {name: partition_records(name) for name in expected_partitions}

    # Pairwise disjointness checks guard against accidental subject or record leakage.
    for left in expected_partitions:
        # Compare against every other partition, not just the "next" one, so every
        # unordered pair is checked exactly once per left/right combination.
        for right in expected_partitions - {left}:
            # A non-empty intersection means the same subject appears in two
            # partitions -- this is the exact leakage bug the record/subject-grouped
            # splitter (splitting.py) exists to prevent; see docs/record-grouped-splitting.md.
            if subject_sets[left] & subject_sets[right]:
                raise ValueError(f"Subject leakage detected between {left} and {right}")
            # A record can only leak if a subject already has (subjects contain
            # records), but this checks record identity directly as a second,
            # independent guarantee rather than relying solely on the subject check above.
            if record_sets[left] & record_sets[right]:
                raise ValueError(f"Record leakage detected between {left} and {right}")

    split_status = split_quality.get("status", "not available")

{
    "step_2_ready": STEP2_READY,
    "schema_version": index.get("schema_version") if index else None,
    "split": f"{index.get('split_name')}@{index.get('split_version')}" if index else None,
    "split_quality_status": split_status,
    "train_records": len(record_sets.get("train", set())),
    "validation_records": len(record_sets.get("validation", set())),
    "test_records_indexed_but_not_opened": len(record_sets.get("test", set())),
}


## 4. Load raw waveform windows

Only train and validation shard descriptors are resolved and opened.

The protected test partition remains indexed for lineage but unopened.

The features are raw waveform window rows stored in each record-level NPZ shard. No FFT augmentation, engineered feature expansion, local checkpoints, or exploratory outputs are used.


> **CODE CELL FUNCTION**
>
> Define file integrity helpers and load only train/validation waveform windows from indexed NPZ shards.


In [ ]:

# COMMENT MAP:
# - Load waveform windows only for train and validation partitions.
# - Refuse protected test loading at the helper boundary.
# - Validate shard hashes, array shapes, finite values, targets, and record lineage.
# - Report one start and one completion line for the complete load phase, not per shard.
# - Initialize empty placeholders in blocked mode so later cells can skip safely.

# These six placeholders are overwritten together by load_partition("train") and
# load_partition("validation") in the ready branch further down this cell.
X_train = None
# Train targets, aligned index-for-index with X_train.
y_train = None
# Train record IDs, aligned index-for-index with X_train's window rows.
train_records = []
# Validation features, loaded the same way as X_train but from the validation partition.
X_validation = None
# Validation targets, aligned index-for-index with X_validation.
y_validation = None
# Validation record IDs, aligned index-for-index with X_validation's window rows.
validation_records = []


def sha256_file(path: Path) -> str:
    """Compute a SHA-256 digest for an indexed shard when expected hashes are present."""
    digest = hashlib.sha256()
    # Stream in fixed-size chunks rather than read_bytes() so hashing a large shard
    # doesn't require holding its entire contents in memory at once.
    with path.open("rb") as source:
        # iter(callable, sentinel) repeatedly calls the lambda until it returns the
        # sentinel b"" (EOF), which is the standard idiom for chunked file reads.
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_indexed_file(shard: dict[str, Any], *, partition: str) -> Path:
    """Resolve and validate one shard path from the dataset index."""
    relative_path = (shard.get("file") or {}).get("path")
    # A missing or empty path means this shard descriptor is malformed at the index
    # level, before any filesystem lookup is even attempted.
    if not isinstance(relative_path, str) or not relative_path:
        raise ValueError(f"Shard in {partition} partition is missing a relative path")

    shard_path = root / relative_path
    require_file(shard_path, repository_root=root, purpose=f"Window shard for {partition} partition is required before loading waveform arrays.")

    expected_sha256 = (shard.get("file") or {}).get("sha256")
    # A digest is only checked when the index actually recorded one; its absence is
    # not itself an error (older index schemas may omit it).
    if expected_sha256:
        observed_sha256 = sha256_file(shard_path)
        # A mismatch means the on-disk shard no longer matches what the index
        # recorded at generation time -- silently loading it would risk training or
        # validating on a corrupted or substituted file.
        if observed_sha256 != expected_sha256:
            raise ValueError(f"SHA-256 mismatch for indexed shard: {shard_path.relative_to(root)}")
    return shard_path


def load_partition(name: str) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Load all waveform windows and integer targets for one non-test partition."""
    # This is the enforcement point for the protected-test-partition boundary: even
    # though partitions[name] resolution below would work for "test" (its metadata
    # is present in the index), this function refuses to ever read its shard files.
    # See docs/evaluation-policy.md and docs/benchmark-governance.md.
    if name == "test":
        raise ValueError("Protected test partition must not be opened by this notebook")

    matrices: list[np.ndarray] = []
    targets: list[np.ndarray] = []
    records: list[str] = []
    partition = partitions[name]

    # Load every shard belonging to this partition; each shard is one record's
    # full set of extracted windows.
    for shard in partition["shards"]:
        shard_path = resolve_indexed_file(shard, partition=name)
        expected_record = str(shard["record_id"])

        # Each record-level shard should contain windows, targets, and record lineage arrays.
        with np.load(shard_path, allow_pickle=False) as artifact:
            windows = np.asarray(artifact["windows"])
            labels = np.asarray(artifact["target_values"])
            record_ids = np.asarray(artifact["record_ids"])

        # Validate shape, dtype, finiteness, label alignment, and lineage before concatenation.
        if windows.ndim != 2 or windows.dtype.kind != "f" or not np.isfinite(windows).all():
            raise ValueError(f"Invalid finite floating-point window matrix: {shard_path.relative_to(root)}")
        # A malformed or misaligned target vector would silently corrupt training
        # labels rather than failing loudly at load time.
        if labels.ndim != 1 or labels.dtype.kind not in {"i", "u"} or len(labels) != len(windows):
            raise ValueError(f"Invalid target vector: {shard_path.relative_to(root)}")
        # Every window row in a shard must belong to the one record this shard was
        # named for; any other record ID present means shard/index mislabeling.
        if set(record_ids.tolist()) != {expected_record}:
            raise ValueError(f"Record lineage mismatch: {shard_path.relative_to(root)}")

        matrices.append(windows.astype(np.float64, copy=False))
        targets.append(labels.astype(np.int64, copy=False))
        records.append(expected_record)

    features = np.concatenate(matrices, axis=0)
    labels = np.concatenate(targets)

    # Compare loaded counts against the dataset index to catch incomplete or mismatched artifacts.
    expected_counts = {str(k): v for k, v in sorted(Counter(labels.tolist()).items())}
    # A class-count mismatch means the concatenated shards don't actually match what
    # the index recorded, even though every individual shard passed its own checks.
    if expected_counts != partition.get("target_value_counts"):
        raise ValueError(f"Loaded {name} targets do not match dataset index counts")
    # A row-count mismatch is a second, independent cross-check against silent
    # partial loads (e.g. a shard that loaded but contributed the wrong row count).
    if features.shape[0] != partition.get("window_count"):
        raise ValueError(f"Loaded {name} window count does not match dataset index")
    return features, labels, records


# Mirrors cells 4 and 5's own STEP2_READY guard: this cell must not attempt to
# open any shard files against artifacts that were never confirmed to exist.
if not STEP2_READY:
    display_markdown("**Waveform loading skipped.** Step 2 is blocked until Step 0 produces train/validation shards.")
else:
    # The range is deliberately qualified: hashing and loading scale with local
    # artifact size and disk speed, so it is guidance rather than a guarantee.
    with NOTEBOOK_PROGRESS.stage(
        "load train/validation waveform shards",
        1,
        3,
        detail="often seconds to a few minutes; artifact size and disk speed vary",
    ) as load_stage:
        X_train, y_train, train_records = load_partition("train")
        X_validation, y_validation, validation_records = load_partition("validation")
        # A single aggregate completion detail confirms useful scale and the
        # protected-test boundary without printing one noisy line per shard.
        load_stage.detail(
            f"{len(X_train):,} train + {len(X_validation):,} validation windows; protected test unopened"
        )

{
    "step_2_ready": STEP2_READY,
    "feature_source": "raw waveform windows only" if STEP2_READY else None,
    "window_samples": int(index["window_samples"]) if STEP2_READY else None,
    "train_shape": tuple(int(value) for value in X_train.shape) if STEP2_READY else None,
    "validation_shape": tuple(int(value) for value in X_validation.shape) if STEP2_READY else None,
    "train_counts": {str(k): int(v) for k, v in Counter(y_train.tolist()).items()} if STEP2_READY else {},
    "validation_counts": {str(k): int(v) for k, v in Counter(y_validation.tolist()).items()} if STEP2_READY else {},
    "test_partition_opened": False,
}


## 5. Train fixed tuned gradient boosting model

This example intentionally uses one fixed configuration derived from prior local experimentation.

It is not a hyperparameter search notebook and does not claim this configuration is universally optimal.

Balanced sample weights are applied during fitting to reduce the effect of class imbalance in the training partition.

The configuration also pins a random seed, `random_state=0`. Gradient boosting with histogram binning has internal randomness, and fixing the seed means repeating the fit on the same generated artifacts reproduces the same model and the same validation metrics below. It is part of the fixed configuration for reproducibility, not a tuned choice.


> **CODE CELL FUNCTION**
>
> Fit one fixed `HistGradientBoostingClassifier` on the train partition using balanced sample weights.


In [ ]:

# COMMENT MAP:
# - Train one fixed HistGradientBoostingClassifier only when train/validation arrays are loaded.
# - Keep the hyperparameters fixed; this is not a search loop or leaderboard optimization.
# - Use balanced sample weights to reduce majority-class dominance during local validation.
# - Report a qualified start, minute-scale heartbeats, and measured completion around fitting.
# - Do not save a model artifact; the repository artifact boundary remains clean.

model = None

# Mirrors the earlier cells' own STEP2_READY guard: this cell must not attempt to
# fit a model against arrays that were never successfully loaded.
if not STEP2_READY:
    raise RuntimeError("Step 2 is not ready; strict prerequisite gate did not pass, so model fitting must not run.")
else:
    # Fixed model configuration: no search loop, no leaderboard tuning, no protected-test feedback.
    model = HistGradientBoostingClassifier(
        learning_rate=0.015,
        max_leaf_nodes=31,
        min_samples_leaf=30,
        l2_regularization=0.1,
        max_iter=450,
        # Fixed seed for the estimator's internal randomness: refitting on the
        # same generated artifacts reproduces the same model and metrics.
        random_state=0,
    )

    # A typical-laptop estimate is intentionally broad and conditional because
    # generated dataset size, CPU, and available memory vary across local runs.
    with NOTEBOOK_PROGRESS.stage(
        "fit fixed gradient boosting model",
        2,
        3,
        detail="often a few minutes on a typical laptop; data size and hardware vary",
    ) as training_stage:
        # Balanced weights reduce the dominance of majority classes during training.
        sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

        # Minute-scale elapsed heartbeats keep a multi-minute library call visibly active
        # without enabling noisy per-iteration estimator logs or predicting completion.
        with NOTEBOOK_PROGRESS.heartbeat(
            "[2/3] fit fixed gradient boosting model",
            interval_seconds=60.0,
        ):
            # The model is trained in memory only. No trained model is written to artifacts/.
            model.fit(X_train, y_train, sample_weight=sample_weight)
        # Report only aggregate workload context; do not expose per-iteration noise
        # or turn measured local timing into a portability claim.
        training_stage.detail(f"{len(X_train):,} training windows; in-memory model only")

{
    "step_2_ready": STEP2_READY,
    "estimator": model.__class__.__name__ if model is not None else None,
    "learning_rate": model.learning_rate if model is not None else None,
    "max_leaf_nodes": model.max_leaf_nodes if model is not None else None,
    "min_samples_leaf": model.min_samples_leaf if model is not None else None,
    "l2_regularization": model.l2_regularization if model is not None else None,
    "max_iter": model.max_iter if model is not None else None,
    "sample_weighting": "balanced" if model is not None else None,
    "model_saved_as_repository_artifact": False,
}


## 6. Validation-only metrics

The next cell scores only the validation partition.

<div role="note" aria-label="Validation evidence limitation" style="border: 3px solid #007c91; border-left-width: 8px; border-radius: 6px; background: #eefbff; color: #102a43; padding: 1rem 1.25rem; margin: 1rem 0; line-height: 1.5;"><strong style="display: block; font-size: 1.15rem; margin-bottom: 0.35rem;">Validation evidence only</strong>These are observed local validation-only results for the grouped validation partition. Under the <a href="../docs/evaluation-policy.md">evaluation policy</a>, they are not benchmark evidence, clinical evidence, protected-test evidence, or production model performance.</div>


> **CODE CELL FUNCTION**
>
> Predict validation labels and display precision, recall, F1, support, and accuracy as a Markdown table.


In [ ]:

# COMMENT MAP:
# - Evaluate only the validation partition.
# - Derive metric labels from train plus validation labels to keep zero-support classes visible.
# - Render a Markdown table for public readability.
# - Report one start/completion pair around prediction and metric calculation.
# - Avoid opening or referencing protected test labels.

# All eight placeholders below are overwritten together in the ready branch further
# down this cell; they exist only so blocked mode has defined names to report.
validation_predictions = None
# Sorted label set the per-class metrics table below is indexed by.
classes = ()
# Overall fraction of correct predictions across all validation classes.
accuracy = None
# Aggregate metrics unweighted by class support (each class counts equally).
macro_precision = None
# Unweighted mean recall across classes; see macro_precision above.
macro_recall = None
# Unweighted mean F1 across classes; see macro_precision above.
macro_f1 = None
# Aggregate metrics weighted by each class's support (frequency in validation).
weighted_precision = None
# Support-weighted mean recall; see weighted_precision above.
weighted_recall = None
# Support-weighted mean F1; see weighted_precision above.
weighted_f1 = None

# Mirrors the earlier cells' own STEP2_READY guard: this cell must not score
# predictions against a model that was never successfully fit.
if not STEP2_READY:
    raise RuntimeError("Step 2 is not ready; strict prerequisite gate did not pass, so validation metrics must not run.")
else:
    # Validation scoring is normally shorter than fitting, but the deliberately
    # broad range avoids promising a runtime independent of local data/hardware.
    with NOTEBOOK_PROGRESS.stage(
        "score validation partition",
        3,
        3,
        detail="often seconds to a few minutes; validation size and hardware vary",
    ) as validation_stage:
        # Validation is the only evaluation partition opened by this notebook.
        validation_predictions = model.predict(X_validation)

        # Classes are derived from train plus validation labels so zero-support edge cases remain visible.
        classes = tuple(int(value) for value in sorted(set(y_train.tolist()) | set(y_validation.tolist())))

        precision, recall, f1, support = precision_recall_fscore_support(
            y_validation,
            validation_predictions,
            labels=list(classes),
            zero_division=0,
        )
        accuracy = accuracy_score(y_validation, validation_predictions)
        macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
            y_validation,
            validation_predictions,
            average="macro",
            zero_division=0,
        )
        weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
            y_validation,
            validation_predictions,
            average="weighted",
            zero_division=0,
        )

        rows = [
            "| Class | Precision | Recall | F1 | Support |",
            "|---:|---:|---:|---:|---:|",
        ]
        # zip(..., strict=True) pairs each class's precision/recall/F1/support into one
        # table row; strict=True catches a length mismatch as a loud error instead of
        # silently truncating to the shortest array.
        for label, p, r, score, count in zip(classes, precision, recall, f1, support, strict=True):
            rows.append(f"| {label} | {p:.4f} | {r:.4f} | {score:.4f} | {int(count)} |")
        rows.extend([
            f"| **Macro avg** | **{macro_precision:.4f}** | **{macro_recall:.4f}** | **{macro_f1:.4f}** | **{len(y_validation)}** |",
            f"| **Weighted avg** | **{weighted_precision:.4f}** | **{weighted_recall:.4f}** | **{weighted_f1:.4f}** | **{len(y_validation)}** |",
            f"| **Accuracy** |  |  | **{accuracy:.4f}** | **{len(y_validation)}** |",
        ])

        display_markdown("\n".join(rows))
        # Completion detail confirms scope without adding another results surface;
        # the Markdown table remains the only metric presentation in this cell.
        validation_stage.detail(f"{len(X_validation):,} validation windows; protected test unopened")


## 7. Confusion matrix

Rows are true validation labels. Columns are predicted labels.

This figure is a notebook visualization only. It is not saved as a repository artifact by this notebook.


> **CODE CELL FUNCTION**
>
> Display a validation-only confusion matrix for local inspection.


In [ ]:

# COMMENT MAP:
# - Plot the validation-only confusion matrix when predictions exist.
# - Keep the figure local to the notebook; no plot artifact is written to the repository.
# - Annotate each cell so the chart remains interpretable in static notebook views.
# - Skip cleanly when Step 2 is blocked before model fitting.

if not STEP2_READY:
    raise RuntimeError("Step 2 is not ready; validation predictions do not exist, so confusion matrix must not run.")
else:
    # Confusion matrix uses validation labels and validation predictions only.
    matrix = confusion_matrix(y_validation, validation_predictions, labels=list(classes))

    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(matrix)
    ax.set_title("Validation-only confusion matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(range(len(classes)), classes)
    ax.set_yticks(range(len(classes)), classes)

    # Annotate each cell so the plot remains readable without hovering or external files.
    for row in range(matrix.shape[0]):
        # Iterate every column for the current row so each matrix cell gets exactly
        # one centered count annotation.
        for column in range(matrix.shape[1]):
            ax.text(column, row, str(int(matrix[row, column])), ha="center", va="center")

    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()


## 8. Baseline comparison when available

If the same pipeline run includes deterministic baseline validation metrics, this cell compares the local gradient boosting validation result to that repository baseline artifact.

Missing baseline metrics are handled explicitly so the notebook remains usable when only processed dataset artifacts exist.


> **CODE CELL FUNCTION**
>
> Load optional repository baseline validation metrics and compare them with the in-memory gradient boosting result.


In [ ]:

# COMMENT MAP:
# - Compare against optional baseline validation metrics only when available.
# - Treat missing baseline metrics as an informational condition, not a failure.
# - Keep comparison strictly validation-only.
# - Skip cleanly when Step 2 is blocked before model metrics exist.

def baseline_summary(path: Path) -> dict[str, float] | None:
    """Load optional baseline validation metrics from the governed pipeline run."""
    # Baseline metrics are genuinely optional (see this cell's own COMMENT MAP
    # above); a missing path/file is an informational "no baseline" state, not an error.
    if not path or not path.is_file():
        return None
    metrics = load_json(path)
    macro = metrics.get("macro_average", {})
    return {
        "accuracy": float(metrics.get("accuracy")),
        "macro_precision": float(macro.get("precision")),
        "macro_recall": float(macro.get("recall")),
        "macro_f1": float(macro.get("f1")),
    }


# Mirrors the earlier cells' own STEP2_READY guard: this cell must not compare
# against metrics that were never successfully computed.
if not STEP2_READY:
    raise RuntimeError("Step 2 is not ready; validation metrics do not exist, so baseline comparison must not run.")
else:
    baseline = baseline_summary(baseline_metrics_path)
    gradient_boosting = {
        "accuracy": float(accuracy),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
    }

    # No baseline is an expected, non-fatal state (see baseline_summary's own
    # comment above), so this reports plainly rather than raising.
    if baseline is None:
        display_markdown("Baseline validation metrics were not found for this run, so no baseline comparison is shown.")
    else:
        rows = [
            "| Model | Accuracy | Macro precision | Macro recall | Macro F1 |",
            "|---|---:|---:|---:|---:|",
            f"| Repository deterministic baseline | {baseline['accuracy']:.4f} | {baseline['macro_precision']:.4f} | {baseline['macro_recall']:.4f} | {baseline['macro_f1']:.4f} |",
            f"| Fixed HistGradientBoosting local example | {gradient_boosting['accuracy']:.4f} | {gradient_boosting['macro_precision']:.4f} | {gradient_boosting['macro_recall']:.4f} | {gradient_boosting['macro_f1']:.4f} |",
        ]
        display_markdown("\n".join(rows))


## 9. Lineage and limitations

The trained estimator in this notebook is an in-memory local experiment. It is not saved as a supported model artifact, not added to the [run manifest](../docs/run-manifests.md), and not promoted to benchmark status.

Interpretation boundaries:

- Inputs are generated local pipeline artifacts under ignored `data/` and `artifacts/` paths.
- Features are raw waveform windows only.
- Fitting uses the train partition only.
- Evaluation uses the validation partition only.
- The protected test partition remains unopened and unreported.
- Observed values are local experiment results, not benchmark evidence.
- Results are not clinical evidence, diagnostic evidence, deployment fitness, or medical utility.
- LightGBM remains an optional future local optimization candidate, not a default public notebook dependency.

<div role="note" aria-label="Interpretation limitations" style="border: 3px solid #007c91; border-left-width: 8px; border-radius: 6px; background: #eefbff; color: #102a43; padding: 1rem 1.25rem; margin: 1rem 0; line-height: 1.5;"><strong style="display: block; font-size: 1.15rem; margin-bottom: 0.35rem;">Interpretation limitations</strong>This notebook does not establish clinical validity, diagnostic value, medical utility, deployment fitness, benchmark performance, or generalization to unseen populations. Return to the <a href="./01-narrative-walkthrough.ipynb">Step 1 narrative walkthrough</a> for the complete modernization context.</div>


> **CODE CELL FUNCTION**
>
> Summarize run lineage, model boundaries, validation-only metrics, and claim limitations in a compact dictionary.


In [ ]:

# COMMENT MAP:
# - Summarize lineage and claim boundaries whether Step 2 is ready or blocked.
# - Avoid undefined variables by using guarded expressions for run-specific values.
# - Preserve the explicit non-benchmark, non-clinical, non-production claim boundary.
# - Confirm again that the protected test partition was not opened.

if STEP2_READY:
    summary = {
        "step_2_status": "completed_validation_example",
        "run_id": run_id,
        "dataset_index": str(index_path.relative_to(root)),
        "split": f"{index.get('split_name')}@{index.get('split_version')}",
        "mapping": f"{index.get('mapping_name')}@{index.get('mapping_version')}",
        "windowing": f"{index.get('window_config_name')}@{index.get('window_config_version')}",
        "sample_rate_hz": index.get("sample_rate_hz"),
        "channel": {"index": index.get("channel_index"), "name": index.get("channel_name")},
        "features": "raw waveform windows only",
        "estimator": "HistGradientBoostingClassifier",
        "training_partition": "train",
        "evaluation_partition": "validation",
        "test_partition_opened": False,
        "model_saved_as_repository_artifact": False,
        "validation_accuracy": float(accuracy),
        "validation_macro_f1": float(macro_f1),
        "claim_boundary": "local validation-only experiment; not benchmark, clinical, or production evidence",
    }
else:
    summary = {
        "step_2_status": "blocked_before_model_training",
        "blockers": STEP2_BLOCKERS,
        "warnings": STEP2_WARNINGS,
        "context": STEP2_CONTEXT,
        "training_partition": None,
        "evaluation_partition": None,
        "test_partition_opened": False,
        "model_saved_as_repository_artifact": False,
        "claim_boundary": "blocked local validation-only example; no benchmark, clinical, or production evidence",
        "required_next_step": STEP_0_NOTEBOOK,
    }

summary


## Appendix: Version history and change log

These maintenance notes remain in reverse chronology for auditability without interrupting the validation workflow above.

| Version | Change |
|---:|---|
| 3.88 | Fixed the local-checkout kernel guard so it accepts uv's symlinked `.venv` interpreter: it now compares the interpreter path lexically like Step 0 instead of resolving the symlink to the base Python, which had falsely rejected the correctly selected kernel. |
| 3.87 | Documented the fixed `random_state=0` seed: it pins the estimator's internal randomness so repeated fits on the same generated artifacts reproduce the same model and validation metrics. |
| 3.86 | Restored the supported workflow to one local VS Code/Jupyter checkout and locked project kernel. The opening cell now confirms local continuity without copying artifacts, and the CLI probe always uses the locked local environment; optional web-hosted integration is deferred to Issue #200. |
| 3.85 | Superseded by 3.86: explored a cross-runtime continuity handoff during review; that unvalidated path is not part of the supported notebook workflow. |
| 3.84 | Added a conversational overview, qualified first-run/rerun timing, and compact jump links; moved this history to an appendix; made repository-path success visible; and added restrained minute-scale heartbeats during the unchanged model fit. |
| 3.83 | Added concise start/completion feedback for waveform loading, model fitting, and validation scoring, with qualified runtime expectations and measured elapsed times. Model configuration, evaluation boundaries, results, and artifact policy are unchanged. |
| 3.82 | Added repository-relative workflow and policy links plus accessible callout panels for prerequisites, use boundaries, validation evidence, and interpretation limits. |
| 3.81 | Removed stale non-throwing prerequisite language from the public Step 2 description so the notebook contract matches strict validation-example behavior. |
| 3.80 | Restored strict validation-example semantics. Step 2 no longer treats missing Step 0 artifacts or missing modeling dependencies as a successful skipped notebook path; it fails fast until the public workflow can actually load governed artifacts, train, and evaluate on validation. |
| 3.77 | Fixed blocked-mode rendering when modeling dependencies are missing. Step 2 now defines a `display_markdown` helper before optional modeling imports, so missing `sklearn` reports as controlled notebook UX instead of `TypeError: NoneType object is not callable`. |

### Earlier detailed history

| Version | Change | Rationale |
|---|---|---|
| 3.76 | Moved the public Step 2 notebook to the notebook root path and updated Step 0 / Step 1 links accordingly | Keeps the public workflow in one visible ordered sequence instead of hiding the model example under the nested examples directory |
| 3.76 | Converted Step 2 prerequisite failures into controlled blocked-state UX | Prevents a known Step 0 / PIPE-006 blocked state from turning into a notebook traceback during public review |
| 3.76 | Added explicit `COMMENT MAP` blocks and verbose internal comments to every Python code cell | Makes control flow, mutation boundaries, and validation intent auditable before execution |
| 3.75 | Split environment setup, launch guidance, local artifact preflight, and governed pipeline execution into a dedicated Step 0 setup notebook | Keeps the model notebook focused on validation-only training and evaluation |
| 3.75 | Added a streamlined prerequisite validation suite at the top of the model notebook | Fails elegantly and directs users to Step 0 if required environment or generated artifacts are missing |
| 3.5.2 | Expanded Section 1.1 launcher cells into larger opt-in executable templates with `printf` status updates, step labels, elapsed-time reporting, and clearer no-op behavior when left commented/disabled | Makes setup activity visible during environment preparation while still preventing accidental installs, dependency syncs, or nested JupyterLab launches |
| 3.5.2 | Reordered the change log in reverse chronology | Keeps the most recent notebook governance and UX changes at the top without dropping earlier version history |
| 3.5.1 | Converted the local terminal launcher from markdown fenced code blocks into opt-in commented `%%bash` code cells with explicit `**INSTRUCTIONS**` guidance | Keeps copy/paste setup commands in executable notebook cells while preventing accidental environment mutation or nested JupyterLab launches |
| 3.5 | Explored alternative notebook launch environments during early development; those options are historical and superseded by the supported local workflow in 3.86 | Preserves the development record without representing those environments as current support |
| 3.5 | Added optional environment bootstrap, runtime diagnostics, and guarded launcher-script creation cells | Improves reviewer onboarding without automatically modifying the repository or weakening artifact governance |
| 3.0 | Added defensive preflight checks for manifestless raw files, interrupted acquisition directories, and stale generated artifacts | Converts common local reproducibility failures into actionable remediation guidance before artifact loading |
| 3.0 | Added targeted pipeline-failure interpretation for acquisition-manifest errors and inconsistent window-shard identity errors | Helps users recover from partial local runs without hiding the underlying reproducibility guardrails |
| 3.0 | Added safe clean-rebuild remediation commands while avoiding automatic destructive cleanup | Keeps remediation explicit and user-controlled |
| 2.0 | Added first-run prerequisite guidance, governed pipeline command, and `**CODE CELL FUNCTION**` tags before code cells | Made the notebook more usable for reviewers opening it from a fresh clone |
| 2.0 | Added clearer comments across setup, artifact loading, split validation, model fitting, metrics, and lineage cells | Improved scanability and reduced hidden assumptions |
| 2.0 | Cleared execution counts and outputs | Prevented committed local execution state, generated metrics, and prior failures from appearing in the public notebook |
| 1.0 | Initial public high-performing gradient boosting validation notebook | Demonstrated a fixed classical ML validation-only example using generated pipeline artifacts |
